# Candidate Elimination Algorithm - Starter Notebook
Maintains a version space (S boundary + G boundary) and eliminates hypotheses inconsistent with examples.

In [ ]:
import pandas as pd
from copy import deepcopy

In [ ]:
columns = ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast', 'EnjoySport']
data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same',   'Yes'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Warm', 'Same',   'Yes'],
    ['Rainy', 'Cold', 'High',   'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Cool', 'Change', 'Yes'],
]
df = pd.DataFrame(data, columns=columns)
FEATURES = columns[:-1]
n = len(FEATURES)

In [ ]:
def is_more_general(h1, h2):
    """True if h1 is strictly more general than h2."""
    more_gen = False
    for a, b in zip(h1, h2):
        if a == '0' and b != '0':
            return False
        if a != b and a != '?':
            return False
        if a == '?' and b != '?':
            more_gen = True
    return more_gen

def consistent(h, x):
    return all(hi == '?' or hi == xi for hi, xi in zip(h, x))

def candidate_elimination(df, features, target='EnjoySport'):
    S = [['0'] * len(features)]                   # most specific boundary
    G = [['?'] * len(features)]                   # most general boundary

    for _, row in df.iterrows():
        x = list(row[features])
        label = row[target]

        if label == 'Yes':
            G = [g for g in G if consistent(g, x)]
            new_S = []
            for s in S:
                if not consistent(s, x):
                    # Minimally generalise s
                    for i in range(len(s)):
                        if s[i] == '0':
                            s_new = s.copy()
                            s_new[i] = x[i]
                            if any(is_more_general(g, s_new) for g in G):
                                new_S.append(s_new)
                        elif s[i] != x[i]:
                            s_new = s.copy()
                            s_new[i] = '?'
                            if any(is_more_general(g, s_new) for g in G):
                                new_S.append(s_new)
                else:
                    new_S.append(s)
            S = new_S if new_S else S
        else:
            S = [s for s in S if not consistent(s, x)]
            new_G = []
            for g in G:
                if consistent(g, x):
                    # Minimally specialise g
                    for i in range(len(g)):
                        if g[i] == '?':
                            for val in set(df[features[i]]):
                                if val != x[i]:
                                    g_new = g.copy()
                                    g_new[i] = val
                                    if any(is_more_general(g_new, s) for s in S):
                                        new_G.append(g_new)
                else:
                    new_G.append(g)
            G = new_G if new_G else G

        print(f'After example {x} ({label}):')
        print(f'  S = {S}')
        print(f'  G = {G}')

    return S, G

S_final, G_final = candidate_elimination(df, FEATURES)

In [ ]:
print('\n=== Final Version Space ===')
print('S boundary (most specific):', S_final)
print('G boundary (most general): ', G_final)